# Huấn luyện PhoBERT / ViSoBERT trên Colab (Task 9)

Runtime → Change runtime type → **T4 GPU**. Chạy lần lượt từ trên xuống. Mỗi cấu hình ghi `results/runs/*.json`; sao lưu lên Drive sau mỗi lần chạy (ô cuối). **Không** dùng `--eval-test` ở đây (Task 11 mới chạy Test, một lần).

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 2. Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/nlp-absa-visfd"
!mkdir -p {DRIVE}

## 3. Lấy code

In [ ]:
REPO_URL = "https://github.com/CungDoHaiPhong279/nlp-absa-visfd.git"
%cd /content
!git clone -q {REPO_URL} repo || (cd repo && git pull -q)
%cd /content/repo
!git log --oneline -1

## 4. Cài thư viện

In [ ]:
!pip install -q -e . -r requirements.txt
!pip freeze > requirements.lock.colab.txt

## 5. Java (VnCoreNLP cần Java 17+)

In [ ]:
!java -version || (apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless)
!java -version

## 6. Tải dữ liệu

In [ ]:
!python scripts/download_data.py

## 7. Khôi phục cache tách từ từ Drive (nếu có)

`results/runs` đã nằm trong git (D13), không cần khôi phục.

In [ ]:
!mkdir -p data/processed
!cp -n {DRIVE}/data_processed/* data/processed/ 2>/dev/null; ls data/processed | head

## 8. Chạy test

In [ ]:
!pytest -q

## 9. Kiểm tra nạp được tokenizer

In [ ]:
from transformers import AutoTokenizer

for name in ("vinai/phobert-base-v2", "uitnlp/visobert"):
    print(name, AutoTokenizer.from_pretrained(name).tokenize("pin trâu, camera đẹp"))

## 10. Đo thời gian (Task 9 Step 8)

500 dòng, 1 epoch. Ghi thời gian và VRAM (lệnh `nvidia-smi` ngay sau) vào `docs/decisions.md`, rồi ước lượng một lần chạy đầy đủ. Hơn 60 phút/lần: giảm `max_length` trước, rồi mới tăng `batch_size` lên 32.

In [ ]:
!time python scripts/run.py --config configs/phobert_joint.yaml --limit 500 --set model.epochs=1
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 11. Chạy đầy đủ, seed 42 (Task 9 Step 9)

Mỗi ô một cấu hình. Chạy xong ô nào thì chạy ô sao lưu (mục 12) ngay, phòng Colab ngắt.

In [ ]:
!python scripts/run.py --config configs/phobert_joint.yaml

In [ ]:
!python scripts/run.py --config configs/ablation/E6_phobert_split.yaml

In [ ]:
!python scripts/run.py --config configs/ablation/E4_phobert_weight.yaml

In [ ]:
!python scripts/run.py --config configs/ablation/E3_phobert_strip_emoji.yaml

In [ ]:
!python scripts/run.py --config configs/ablation/E5_phobert_star.yaml

In [ ]:
!python scripts/run.py --config configs/visobert_joint.yaml

## 12. Sao lưu lên Drive

Sau đó tải các file `results/runs/*.json` mới về máy, chạy `python scripts/make_tables.py` và mở PR.

In [ ]:
import datetime

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
!mkdir -p {DRIVE}/data_processed {DRIVE}/backups/{stamp} {DRIVE}/models
!cp -r results {DRIVE}/backups/{stamp}/
!cp -n data/processed/* {DRIVE}/data_processed/ 2>/dev/null
!cp -r models/final_* {DRIVE}/models/ 2>/dev/null; ls {DRIVE}/backups/{stamp}/results/runs | tail